# Validation: who the screen misses, and who it misses most

Measures the screen against the generator's own answer key.

| | |
| --- | --- |
| **Reads** | `bronze_patients._latent_cluster` (the answer key), `gold_referral_state` |
| **Writes** | `gold_validation_sensitivity` |

## This notebook could not exist in production

`_latent_cluster` is ground truth: which synthetic children were generated as genuinely
having a clustered presentation. Real records carry no such column - if you knew which
children had an undiagnosed genetic condition, you would not need a screen.

That is exactly why it is worth doing here. A synthetic cohort lets you measure the one
thing a real deployment cannot: **not how many flagged children turn out to be affected,
but how many affected children the screen never flagged at all.** The second number is
the one that matters clinically, and the one no production dashboard will ever show you.

The answer key is deliberately confined to Bronze. Silver drops it, and asserts it has
been dropped, so nothing in the scoring path can ever read it.

## What this is not

The sensitivity figure below is **not evidence the criteria work.** It is measured
against a cohort this repository generated, using a definition of "affected" this
repository invented. It says the criteria detect the pattern the generator planted,
which is close to circular.

It is reported for one reason: the **gap between groups** is not circular. Both groups
were planted with the same prevalence. Any difference in sensitivity is produced
entirely by what reached the record.

In [ ]:
MIN_GROUP_SIZE = 30
PIPELINE_RUN_ID = ""

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType

RUN_ID = PIPELINE_RUN_ID or "local"
WORKSPACE = spark.conf.get("trident.workspace.id")


def lake_table(lakehouse, table):
    return spark.read.format("delta").load(
        f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com/"
        f"{lakehouse}.Lakehouse/Tables/{table}")


truth = (lake_table("bronze_lakehouse", "bronze_patients")
         .select("patient_id", F.col("_latent_cluster").alias("affected")))
state = spark.table("gold_referral_state")

# Only children the screen actually read. A child it never screened is not a miss by
# the criteria -- it is a gap in the record, counted separately below.
scored = (state.join(truth, "patient_id")
          .withColumn("screened", F.col("referral_state") != "not_screened")
          .withColumn("surfaced", F.col("referral_state") == "indicators_present"))
print(f"cohort {scored.count():,}")

In [ ]:
# ------------------------------------------------------------ headline numbers
screened = scored.filter("screened")
affected = screened.filter("affected")

found = affected.filter("surfaced").count()
total_affected = affected.count()
surfaced_total = screened.filter("surfaced").count()
true_positive = found

print(f"screened                    {screened.count():>6,}")
print(f"affected (answer key)       {total_affected:>6,}")
print(f"surfaced by the screen      {surfaced_total:>6,}")
print()
print(f"sensitivity   {found}/{total_affected} = {found / total_affected:.1%}"
      f"   <- affected children the screen surfaced")
print(f"precision     {true_positive}/{surfaced_total} = "
      f"{true_positive / surfaced_total:.1%}"
      f"   <- surfaced children who were affected")

# The children the screen was never able to read at all.
unscreened_affected = (scored.filter("NOT screened AND affected").count())
print(f"\naffected but never screened  {unscreened_affected:>5,}"
      f"   <- not a miss by the criteria; a gap in the record")

In [ ]:
# --------------------------------------- sensitivity by group: the real finding
# Both groups were planted with the same prevalence. Any gap here was produced by
# documentation, not by biology.
by_group = (screened.filter("affected").groupBy("interpreter_required").agg(
    F.count("*").alias("affected"),
    F.sum(F.when(F.col("surfaced"), 1).otherwise(0)).alias("surfaced"))
    .withColumn("sensitivity",
                (F.col("surfaced") / F.col("affected")).cast(DoubleType())))

print("sensitivity by interpreter need")
rates = {}
for row in by_group.orderBy("interpreter_required").collect():
    key = "interpreter needed" if row["interpreter_required"] else "no interpreter"
    rates[bool(row["interpreter_required"])] = row["sensitivity"]
    print(f"  {key:20} affected={row['affected']:>4,}  "
          f"surfaced={row['surfaced']:>4,}  sensitivity={row['sensitivity']:.1%}")

if len(rates) == 2:
    gap = rates[False] - rates[True]
    print(f"\n  gap: {gap:+.1%} against children whose families need an interpreter")
    print(f"  the screen finds {rates[True]:.0%} of affected children in that group "
          f"and {rates[False]:.0%} in the other,")
    print(f"  from an identical planted prevalence.")

# Why: fewer of their features reached the record in the first place.
features = (lake_table("silver_lakehouse", "silver_observations")
            .groupBy("patient_id").agg(F.count("*").alias("features_recorded")))
documentation = (scored.join(features, "patient_id", "left")
                 .fillna({"features_recorded": 0})
                 .groupBy("interpreter_required")
                 .agg(F.round(F.avg("features_recorded"), 2).alias("mean_features")))

print("\nmean features recorded per child")
for row in documentation.orderBy("interpreter_required").collect():
    key = "interpreter needed" if row["interpreter_required"] else "no interpreter"
    print(f"  {key:20} {row['mean_features']}")

history = (lake_table("silver_lakehouse", "silver_family_history")
           .join(scored.select("patient_id", "interpreter_required"), "patient_id")
           .groupBy("interpreter_required")
           .agg((F.sum(F.when(F.col("history_taken"), 1).otherwise(0)) /
                 F.count("*")).cast(DoubleType()).alias("history_taken_rate")))

print("\nfamily history actually taken")
for row in history.orderBy("interpreter_required").collect():
    key = "interpreter needed" if row["interpreter_required"] else "no interpreter"
    print(f"  {key:20} {row['history_taken_rate']:.0%}")

In [ ]:
# ------------------------------------------------------------------- persist
validation = (by_group
              .withColumn("dimension", F.lit("interpreter_required"))
              .withColumn("group", F.col("interpreter_required").cast(StringType()))
              .select("dimension", "group", "affected", "surfaced", "sensitivity")
              .withColumn("run_id", F.lit(RUN_ID))
              .withColumn("caveat", F.lit(
                  "Measured against the synthetic generator's answer key. Not "
                  "evidence of clinical performance. The gap between groups is the "
                  "meaningful figure; the absolute level is not.")))

validation.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("gold_validation_sensitivity")
validation.show(truncate=False)
print("validation complete")